# Election2024 Twitter Dataset Access Testing

In [1]:
import os
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Change path on .env file on your local machine
dataset_path = Path(os.getenv("DATASET_PATH"))
#output_dir = Path(os.getenv("OUTPUT_DIR"))


In [2]:
print(dataset_path)

C:\Users\andre\OneDrive\Documents\GitHub\Thesis_Repository\Elec24 Test\parts_initial_test


In [3]:
import importlib
import utils

importlib.reload(utils)
from utils import load_chunks_by_part, load_sample_chunks, summarize_part_schemas

## Initial Cleaning of the Dataset

Initially clean the first two parts of the dataset so to start practicing the susceptibility paradox codes in computing the IAR/SAR values needed

In [4]:
# TODO: Access the dataset parts and check if their schemas match each other 
# This should match each other as per the test above.
parts_to_clean = [1, 2]
part_dirs = [dataset_path / f"part_{n}" for n in parts_to_clean]
part_dirs = [p for p in part_dirs if p.is_dir()]

if not part_dirs:
    raise FileNotFoundError(
        f"Could not find part_1 or part_2 under {dataset_path}. "
        "Check DATASET_PATH in your .env file."
    )

chunk_schemas = []
all_columns = []

for part_dir in part_dirs:
    for chunk_name, chunk_df in load_chunks_by_part(part_dir):
        columns = list(chunk_df.columns)
        dtypes = {col: str(dtype) for col, dtype in chunk_df.dtypes.items()}
        chunk_schemas.append({
            "part": part_dir.name,
            "chunk": chunk_name,
            "n_columns": len(columns),
            "columns": columns,
            "dtypes": dtypes,
        })
        for col in columns:
            if col not in all_columns:
                all_columns.append(col)

schema_report = pd.DataFrame([
    {
        "part": item["part"],
        "chunk": item["chunk"],
        "n_columns": item["n_columns"],
        "matches_first_chunk_columns": item["columns"] == chunk_schemas[0]["columns"],
        "matches_first_chunk_dtypes": item["dtypes"] == chunk_schemas[0]["dtypes"],
    }
    for item in chunk_schemas
])

display(schema_report)

reference = chunk_schemas[0]
for item in chunk_schemas[1:]:
    if item["columns"] != reference["columns"]:
        print(f"Column differences: {item['part']}/{item['chunk']}")
        print("  Missing:", [c for c in reference["columns"] if c not in item["columns"]])
        print("  Extra:", [c for c in item["columns"] if c not in reference["columns"]])
        if set(item["columns"]) == set(reference["columns"]):
            print("  The columns are present but in a different order.")
    if item["dtypes"] != reference["dtypes"]:
        mismatches = {
            c: (reference["dtypes"].get(c), item["dtypes"].get(c))
            for c in set(reference["dtypes"]) | set(item["dtypes"])
            if reference["dtypes"].get(c) != item["dtypes"].get(c)
        }
        print(f"Data type differences: {item['part']}/{item['chunk']}: {mismatches}")

print(f"Checked {len(chunk_schemas)} chunks across {[p.name for p in part_dirs]}.")
print(f"Union of columns: {len(all_columns)}")

  Loading chunk: may_july_chunk_1.csv.gz
  Loading chunk: may_july_chunk_10.csv.gz
  Loading chunk: may_july_chunk_11.csv.gz
  Loading chunk: may_july_chunk_12.csv.gz
  Loading chunk: may_july_chunk_13.csv.gz
  Loading chunk: may_july_chunk_14.csv.gz
  Loading chunk: may_july_chunk_15.csv.gz
  Loading chunk: may_july_chunk_16.csv.gz
  Loading chunk: may_july_chunk_17.csv.gz
  Loading chunk: may_july_chunk_18.csv.gz
  Loading chunk: may_july_chunk_19.csv.gz
  Loading chunk: may_july_chunk_2.csv.gz
  Loading chunk: may_july_chunk_20.csv.gz
  Loading chunk: may_july_chunk_3.csv.gz
  Loading chunk: may_july_chunk_4.csv.gz
  Loading chunk: may_july_chunk_5.csv.gz
  Loading chunk: may_july_chunk_6.csv.gz
  Loading chunk: may_july_chunk_7.csv.gz
  Loading chunk: may_july_chunk_8.csv.gz
  Loading chunk: may_july_chunk_9.csv.gz
  Loading chunk: may_july_chunk_21.csv.gz
  Loading chunk: may_july_chunk_22.csv.gz
  Loading chunk: may_july_chunk_23.csv.gz
  Loading chunk: may_july_chunk_24.csv.gz
 

,part,chunk,n_columns,matches_first_chunk_columns,matches_first_chunk_dtypes
0,part_1,may_july_chunk_1.csv.gz,32,True,True
1,part_1,may_july_chunk_10.csv.gz,32,True,True
2,part_1,may_july_chunk_11.csv.gz,32,True,True
3,part_1,may_july_chunk_12.csv.gz,32,True,False
4,part_1,may_july_chunk_13.csv.gz,32,True,False
5,part_1,may_july_chunk_14.csv.gz,32,True,True
6,part_1,may_july_chunk_15.csv.gz,32,True,True
7,part_1,may_july_chunk_16.csv.gz,32,True,False
8,part_1,may_july_chunk_17.csv.gz,32,True,False
9,part_1,may_july_chunk_18.csv.gz,32,True,True


Data type differences: part_1/may_july_chunk_12.csv.gz: {'replyCount': ('int64', 'float64'), 'retweetCount': ('int64', 'float64')}
Data type differences: part_1/may_july_chunk_13.csv.gz: {'replyCount': ('int64', 'float64'), 'retweetCount': ('int64', 'float64')}
Data type differences: part_1/may_july_chunk_16.csv.gz: {'replyCount': ('int64', 'float64'), 'retweetCount': ('int64', 'float64')}
Data type differences: part_1/may_july_chunk_17.csv.gz: {'replyCount': ('int64', 'float64'), 'retweetCount': ('int64', 'float64')}
Data type differences: part_1/may_july_chunk_5.csv.gz: {'retweetCount': ('int64', 'float64')}
Data type differences: part_1/may_july_chunk_6.csv.gz: {'replyCount': ('int64', 'float64'), 'retweetCount': ('int64', 'float64')}
Data type differences: part_1/may_july_chunk_7.csv.gz: {'replyCount': ('int64', 'float64'), 'retweetCount': ('int64', 'float64')}
Data type differences: part_1/may_july_chunk_8.csv.gz: {'replyCount': ('int64', 'float64'), 'retweetCount': ('int64', 'flo

In [5]:
# TODO: Once verified, create a specific schema for the dataset so that pandas/polars can use it to read the data better
# This also avoids the need to infer the schema (an error as per my (andre) previous push)
id_like_columns = {
    "id", "id_str", "conversationId", "conversationIdStr",
    "retweetedTweetID", "retweetedUserID",
    "in_reply_to_status_id_str", "in_reply_to_user_id_str",
}
text_columns = {
    "text", "rawContent", "url", "lang", "hashtags", "mentionedUsers",
    "links", "location", "cash_app_handle", "user", "username",
    "date", "type", "_type", "media", "viewCount",
    "in_reply_to_screen_name",
}
integer_columns = {"replyCount", "retweetCount"}
float_columns = {"epoch", "likeCount", "quoteCount"}

# Keep the union of discovered columns, excluding CSV index columns.
# This handles schema differences without silently discarding part-specific fields.
clean_columns = [
    c for c in all_columns
    if not (str(c).startswith("Unnamed:") or str(c) == "index")
]

print("Columns in the unified schema:")
print(clean_columns)

Columns in the unified schema:
['id', 'text', 'url', 'epoch', 'media', 'retweetedTweet', 'retweetedTweetID', 'retweetedUserID', 'id_str', 'lang', 'rawContent', 'replyCount', 'retweetCount', 'likeCount', 'quoteCount', 'conversationId', 'conversationIdStr', 'hashtags', 'mentionedUsers', 'links', 'viewCount', 'quotedTweet', 'in_reply_to_screen_name', 'in_reply_to_status_id_str', 'in_reply_to_user_id_str', 'location', 'cash_app_handle', 'user', 'date', '_type', 'type']


In [6]:
# TODO: Combine the chunks of the dataset into a single dataframe and save it as a parquet file for easier access in the future.
# Use polars or make it into a parquet file to be memory efficient 
import pyarrow as pa
import pyarrow.parquet as pq

output_path = dataset_path / "cleaned_first_two_parts.parquet"
parquet_writer = None
rows_written = 0

def normalize_chunk_for_parquet(df):
    """Normalize column names/order and data types before writing a chunk."""
    df = df.copy()

    # Drop duplicated column names safely, keeping the first occurrence.
    df = df.loc[:, ~df.columns.duplicated()]

    # Remove CSV index columns created during earlier exports.
    drop_cols = [
        c for c in df.columns
        if str(c).startswith("Unnamed:") or str(c) == "index"
    ]
    df = df.drop(columns=drop_cols, errors="ignore")

    # Add columns absent from this chunk and enforce a stable order.
    for col in clean_columns:
        if col not in df.columns:
            df[col] = pd.NA
    df = df.reindex(columns=clean_columns)

    for col in df.columns:
        if col in id_like_columns or col in text_columns:
            # Nullable string dtype handles missing values consistently.
            df[col] = df[col].astype("string")
        elif col in integer_columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        elif col in float_columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Float64")
        elif col in {"retweetedTweet", "quotedTweet"}:
            # Preserve nulls while normalizing common true/false representations.
            values = df[col].astype("string").str.strip().str.lower()
            df[col] = values.map({
                "true": True, "1": True, "yes": True,
                "false": False, "0": False, "no": False,
            }).astype("boolean")
        else:
            # Keep unknown/special columns as strings to avoid mixed-type inference.
            df[col] = df[col].astype("string")

    return df

try:
    for part_dir in part_dirs:
        for chunk_name, raw_df in load_chunks_by_part(part_dir):
            cleaned_chunk = normalize_chunk_for_parquet(raw_df)

            # Convert to Arrow and keep the schema consistent across chunks.
            table = pa.Table.from_pandas(cleaned_chunk, preserve_index=False)

            if parquet_writer is None:
                parquet_writer = pq.ParquetWriter(
                    output_path, table.schema, compression="snappy"
                )
            else:
                # Align the table to the writer schema if metadata differs.
                table = table.cast(parquet_writer.schema)
            parquet_writer.write_table(table)
            rows_written += len(cleaned_chunk)
            print(f"Wrote {part_dir.name}/{chunk_name}: {len(cleaned_chunk):,} rows")
finally:
    if parquet_writer is not None:
        parquet_writer.close()

if rows_written:
    print(f"Saved {rows_written:,} rows to: {output_path}")
else:
    print("No rows were written. Check the part directories and chunk loader.")

  Loading chunk: may_july_chunk_1.csv.gz
Wrote part_1/may_july_chunk_1.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_10.csv.gz
Wrote part_1/may_july_chunk_10.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_11.csv.gz
Wrote part_1/may_july_chunk_11.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_12.csv.gz
Wrote part_1/may_july_chunk_12.csv.gz: 49,998 rows
  Loading chunk: may_july_chunk_13.csv.gz
Wrote part_1/may_july_chunk_13.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_14.csv.gz
Wrote part_1/may_july_chunk_14.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_15.csv.gz
Wrote part_1/may_july_chunk_15.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_16.csv.gz
Wrote part_1/may_july_chunk_16.csv.gz: 49,984 rows
  Loading chunk: may_july_chunk_17.csv.gz
Wrote part_1/may_july_chunk_17.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_18.csv.gz
Wrote part_1/may_july_chunk_18.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_19.csv.gz
Wrote part_1/may_july_chunk_19

In [7]:
# TODO: Clean the dataset by removing any unnecessary columns, handling missing values, and ensuring data types are consistent across all parts.
# Don't remove null values in important columns


# Basic initial cleaning:
# - remove exported DataFrame index columns
# - normalize column names (trim whitespace)
# - normalize key data types across all chunks
# - preserve rows with missing values in important fields
# - remove only exact duplicate rows within each chunk
#
# The Parquet file above is the normalized base dataset that creates
# second cleaned file while streaming chunks, so it doesn't  load everything
# into RAM. Exact duplicates across different chunks are not removed here;
# global deduplication can be expensive and should use a stable ID if appropriate.

import pyarrow as pa
import pyarrow.parquet as pq

cleaned_output_path = dataset_path / "cleaned_first_two_parts_basic.parquet"
clean_writer = None
clean_rows_written = 0
duplicate_rows_removed = 0

try:
    for part_dir in part_dirs:
        for chunk_name, raw_df in load_chunks_by_part(part_dir):
            df = raw_df.copy()
            df.columns = [str(c).strip() for c in df.columns]

            # Drop duplicated column names and redundant CSV index columns.
            df = df.loc[:, ~df.columns.duplicated()]
            df = df.drop(
                columns=[
                    c for c in df.columns
                    if c.startswith("Unnamed:") or c == "index"
                ],
                errors="ignore",
            )

            before = len(df)
            df = df.drop_duplicates()
            duplicate_rows_removed += before - len(df)

            df = normalize_chunk_for_parquet(df)

            # Do not drop rows merely because text, user, or date is missing.
            # Missing values in these fields are retained for later decisions.
            table = pa.Table.from_pandas(df, preserve_index=False)

            if clean_writer is None:
                clean_writer = pq.ParquetWriter(
                    cleaned_output_path, table.schema, compression="snappy"
                )
            else:
                table = table.cast(clean_writer.schema)

            clean_writer.write_table(table)
            clean_rows_written += len(df)

            print(
                f"Cleaned {part_dir.name}/{chunk_name}: "
                f"{len(df):,} rows"
            )
finally:
    if clean_writer is not None:
        clean_writer.close()

print(f"Rows written: {clean_rows_written:,}")
print(f"Exact duplicate rows removed within chunks: {duplicate_rows_removed:,}")
print(f"Output: {cleaned_output_path}")

# Optional quick validation of the saved dataset.
if cleaned_output_path.exists():
    check_df = pd.read_parquet(cleaned_output_path)
    print("Saved shape:", check_df.shape)
    print("Missing values in key columns:")
    key_columns = [c for c in ["id", "text", "user", "username", "date"] if c in check_df.columns]
    if key_columns:
        display(check_df[key_columns].isna().sum().rename("missing_count").to_frame())

  Loading chunk: may_july_chunk_1.csv.gz
Cleaned part_1/may_july_chunk_1.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_10.csv.gz
Cleaned part_1/may_july_chunk_10.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_11.csv.gz
Cleaned part_1/may_july_chunk_11.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_12.csv.gz
Cleaned part_1/may_july_chunk_12.csv.gz: 49,998 rows
  Loading chunk: may_july_chunk_13.csv.gz
Cleaned part_1/may_july_chunk_13.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_14.csv.gz
Cleaned part_1/may_july_chunk_14.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_15.csv.gz
Cleaned part_1/may_july_chunk_15.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_16.csv.gz
Cleaned part_1/may_july_chunk_16.csv.gz: 49,984 rows
  Loading chunk: may_july_chunk_17.csv.gz
Cleaned part_1/may_july_chunk_17.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_18.csv.gz
Cleaned part_1/may_july_chunk_18.csv.gz: 50,000 rows
  Loading chunk: may_july_chunk_19.csv.gz
Cleaned pa

,missing_count
id,0
text,0
user,0
date,0
